# 04 - Train TF-IDF + Naive Bayes

Notebook nay train model baseline cho bai toan sentiment analysis.

Pipeline:

1. Doc `clean_reviews.csv`
2. Chia train/test
3. Chuyen text thanh vector bang TF-IDF
4. Train Multinomial Naive Bayes
5. Danh gia model
6. Luu model va vectorizer

## Buoc 1 - Import thu vien

In [ ]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Buoc 2 - Khai bao duong dan

In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "clean_reviews.csv"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"

print("Clean data path:", CLEAN_DATA_PATH)
print("Clean file exists:", CLEAN_DATA_PATH.exists())

## Buoc 3 - Doc du lieu da lam sach

In [ ]:
df = pd.read_csv(CLEAN_DATA_PATH)
df["review_clean"] = df["review_clean"].fillna("")

print("So dong:", df.shape[0])
print("So cot:", df.shape[1])

df.head()

## Buoc 4 - Tach input X va label y

- `X`: text da lam sach
- `y`: nhan sentiment can du doan

In [ ]:
X = df["review_clean"]
y = df["sentiment"]

print("So mau X:", X.shape[0])
print("So nhan y:", y.shape[0])

y.value_counts()

## Buoc 5 - Chia train/test

Dung 80% du lieu de train va 20% de test. `stratify=y` giu ty le sentiment gan giong nhau o train va test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

print("\nTrain sentiment ratio:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTest sentiment ratio:")
print(y_test.value_counts(normalize=True).round(3))

## Buoc 6 - Vector hoa text bang TF-IDF

Model khong doc truc tiep text. TF-IDF chuyen moi review thanh vector so.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.9,
    ngram_range=(1, 2),
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

## Buoc 7 - Train Multinomial Naive Bayes

In [ ]:
model = MultinomialNB()

model.fit(X_train_tfidf, y_train)

## Buoc 8 - Du doan tren tap test

In [ ]:
y_pred = model.predict(X_test_tfidf)

y_pred[:10]

## Buoc 9 - Danh gia bang Accuracy va Classification Report

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy, 4))
print()
print(classification_report(y_test, y_pred))

## Buoc 10 - Tao bang ket qua danh gia

In [ ]:
report_dict = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report_dict).T

report_df

## Buoc 11 - Confusion Matrix

In [ ]:
label_order = ["Negative", "Neutral", "Positive"]
cm = confusion_matrix(y_test, y_pred, labels=label_order)

cm_df = pd.DataFrame(cm, index=label_order, columns=label_order)
cm_df

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

## Buoc 12 - Thu du doan review moi

In [ ]:
sample_reviews = [
    "The food was amazing and the staff were very friendly.",
    "The restaurant was okay, not bad but not special.",
    "The food was cold and the service was terrible.",
]

sample_vectors = vectorizer.transform(sample_reviews)
sample_predictions = model.predict(sample_vectors)

pd.DataFrame({
    "review": sample_reviews,
    "predicted_sentiment": sample_predictions,
})

## Buoc 13 - Luu model, vectorizer va ket qua

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(model, MODELS_DIR / "naive_bayes_model.joblib")
joblib.dump(vectorizer, MODELS_DIR / "tfidf_vectorizer.joblib")
report_df.to_csv(REPORTS_DIR / "classification_report.csv")

plt.figure(figsize=(6, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "confusion_matrix.png", dpi=150)
plt.close()

print("Da luu model vao:", MODELS_DIR)
print("Da luu report vao:", REPORTS_DIR)
print("Da luu confusion matrix vao:", FIGURES_DIR)

## Ket luan nhanh

- TF-IDF + Multinomial Naive Bayes la baseline don gian, phu hop cho phan loai text.
- Vi du lieu lech ve `Positive`, can doc ky F1-score tung lop va confusion matrix.
- Neu lop `Neutral` co ket qua thap, day la han che co the neu trong bao cao.